In [3]:
#pip install pandas

In [32]:
# sqlite3 is the standard Python library for SQLite databases
# pandas is used for reading CSV files and handling tabular data
import sqlite3
import pandas as pd

In [33]:
# Name of the local CSV file
# The file should be located in the same directory as this notebook
csv_file = "netflix_titles.csv"

# SQLite database file
# SQLite will create this file if it does not already exist
db_file = "netflix.db"

# Name of the table to be created in SQLite
table_name = "netflix_titles"


In [34]:
# Read the CSV file into a DataFrame
df = pd.read_csv(csv_file)

# Display the first few rows to verify the data loaded correctly
df.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [35]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')

In [36]:
# To get the dimensions of a DataFrame, use shape as an attribute (without parentheses)
df.shape  # This returns a tuple with (number of rows, number of columns)

# If you want to print the dimensions:
print(f"DataFrame dimensions: {df.shape}")

# If you need the number of rows and columns separately:
rows, columns = df.shape
print(f"Number of rows: {rows}")
print(f"Number of columns: {columns}")

DataFrame dimensions: (8807, 12)
Number of rows: 8807
Number of columns: 12


In [37]:
# Create a connection to the SQLite database
conn = sqlite3.connect(db_file)

# Write the DataFrame to a SQLite table
# if_exists options:
#   'replace' → drop the table if it exists and recreate it
#   'append'  → add data to an existing table
#   'fail'    → raise an error if the table already exists
df.to_sql(
    table_name,
    conn,
    if_exists="replace",
    index=False  # Do not store the DataFrame index as a column
)

# Close the database connection
conn.close()

print("netflix_titles.csv successfully loaded into SQLite.")


netflix_titles.csv successfully loaded into SQLite.


In [38]:
# Reconnect to the SQLite database
conn = sqlite3.connect(db_file)

# SQL query to retrieve a sample of rows
query = f"SELECT * FROM {table_name} LIMIT 10"

# Execute the query and load the result into a DataFrame
pd.read_sql(query, conn)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,None,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,None,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",None,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,None,None,None,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,None,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
5,s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, H...",None,"September 24, 2021",2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",The arrival of a charismatic young priest brin...
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",None,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s..."
8,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV",A talented batch of amateur bakers face off in...
9,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,"September 24, 2021",2021,PG-13,104 min,"Comedies, Dramas",A woman adjusting to life after a loss contend...


In [39]:
# Count how many Movies vs TV Shows exist in the dataset
query = "SELECT type, COUNT(*) AS total FROM netflix_titles GROUP BY type;"
pd.read_sql(query, conn)

,type,total
0,Movie,6131
1,TV Show,2676


In [40]:
# Find the top 10 countries with the most titles
query = """
SELECT country, COUNT(*) AS total
FROM netflix_titles
WHERE country IS NOT NULL
GROUP BY country
ORDER BY total DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,country,total
0,United States,2818
1,India,972
2,United Kingdom,419
3,Japan,245
4,South Korea,199
5,Canada,181
6,Spain,145
7,France,124
8,Mexico,110
9,Egypt,106


In [41]:
# Show the most common ratings (like TV-MA, PG, etc.)
query = """
SELECT rating, COUNT(*) AS total
FROM netflix_titles
GROUP BY rating
ORDER BY total DESC;
"""
pd.read_sql(query, conn)

,rating,total
0,TV-MA,3207
1,TV-14,2160
2,TV-PG,863
3,R,799
4,PG-13,490
5,TV-Y7,334
6,TV-Y,307
7,PG,287
8,TV-G,220
9,NR,80


In [42]:
# List titles released after 2015, newest first
query = """
SELECT title, release_year
FROM netflix_titles
WHERE release_year > 2015
ORDER BY release_year DESC;
"""
pd.read_sql(query, conn)

,title,release_year
0,Blood & Water,2021
1,Ganglands,2021
2,Jailbirds New Orleans,2021
3,Kota Factory,2021
4,Midnight Mass,2021
...,...,...
5651,Yoko,2016
5652,YOM,2016
5653,اشتباك,2016
5654,Yunus Emre,2016


In [43]:
# Find the directors with the most titles in the dataset
query = """
SELECT director, COUNT(*) AS total
FROM netflix_titles
WHERE director IS NOT NULL
GROUP BY director
ORDER BY total DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,director,total
0,Rajiv Chilaka,19
1,"Raúl Campos, Jan Suter",18
2,Suhas Kadav,16
3,Marcus Raboy,16
4,Jay Karas,14
5,Cathy Garcia-Molina,13
6,Youssef Chahine,12
7,Martin Scorsese,12
8,Jay Chapman,12
9,Steven Spielberg,11


In [44]:
# Show the longest duration movies (by minutes)
query = """
SELECT title, duration
FROM netflix_titles
WHERE type = 'Movie'
ORDER BY CAST(SUBSTR(duration, 1, INSTR(duration, ' ') - 1) AS INTEGER) DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,title,duration
0,Black Mirror: Bandersnatch,312 min
1,Headspace: Unwind Your Mind,273 min
2,The School of Mischief,253 min
3,No Longer kids,237 min
4,Lock Your Girls In,233 min
5,Raya and Sakina,230 min
6,Once Upon a Time in America,229 min
7,Sangam,228 min
8,Lagaan,224 min
9,Jodhaa Akbar,214 min


In [45]:
# Show TV shows with the most seasons
query = """
SELECT title, duration
FROM netflix_titles
WHERE type = 'TV Show'
ORDER BY CAST(SUBSTR(duration, 1, INSTR(duration, ' ') - 1) AS INTEGER) DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,title,duration
0,Grey's Anatomy,17 Seasons
1,Supernatural,15 Seasons
2,NCIS,15 Seasons
3,Heartland,13 Seasons
4,COMEDIANS of the world,13 Seasons
5,Red vs. Blue,13 Seasons
6,Trailer Park Boys,12 Seasons
7,Criminal Minds,12 Seasons
8,Cheers,11 Seasons
9,Frasier,11 Seasons


In [46]:
# Count titles grouped by genre/category (listed_in column)
query = """
SELECT listed_in, COUNT(*) AS total
FROM netflix_titles
GROUP BY listed_in
ORDER BY total DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,listed_in,total
0,"Dramas, International Movies",362
1,Documentaries,359
2,Stand-Up Comedy,334
3,"Comedies, Dramas, International Movies",274
4,"Dramas, Independent Movies, International Movies",252
5,Kids' TV,220
6,Children & Family Movies,215
7,"Children & Family Movies, Comedies",201
8,"Documentaries, International Movies",186
9,"Dramas, International Movies, Romantic Movies",180


In [47]:
# Show the 10 most recently added titles
query = """
SELECT title, date_added
FROM netflix_titles
WHERE date_added IS NOT NULL
ORDER BY date_added DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,title,date_added
0,Blood Brothers: Malcolm X & Muhammad Ali,"September 9, 2021"
1,Mighty Raju,"September 9, 2021"
2,Paradise Hills,"September 9, 2021"
3,The Women and the Murderer,"September 9, 2021"
4,Cargo,"September 9, 2020"
5,Cuties,"September 9, 2020"
6,Get Organized with The Home Edit,"September 9, 2020"
7,La Línea: Shadow of Narco,"September 9, 2020"
8,So Much Love to Give,"September 9, 2020"
9,The Social Dilemma,"September 9, 2020"


In [48]:
# Search for titles where a specific actor appears in the cast
query = """
SELECT title, "cast"
FROM netflix_titles
WHERE "cast" LIKE '%Leonardo DiCaprio%';
"""
pd.read_sql(query, conn)

,title,cast
0,Catch Me If You Can,"Leonardo DiCaprio, Tom Hanks, Christopher Walk..."
1,Inception,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio..."
2,Django Unchained,"Jamie Foxx, Christoph Waltz, Leonardo DiCaprio..."
3,Shutter Island,"Leonardo DiCaprio, Mark Ruffalo, Ben Kingsley,..."
4,What's Eating Gilbert Grape,"Johnny Depp, Leonardo DiCaprio, Juliette Lewis..."
5,Before the Flood,Leonardo DiCaprio
6,Gangs of New York,"Leonardo DiCaprio, Daniel Day-Lewis, Cameron D..."
7,Revolutionary Road,"Leonardo DiCaprio, Kate Winslet, Kathy Bates, ..."
8,The Departed,"Leonardo DiCaprio, Matt Damon, Jack Nicholson,..."


In [49]:
# Find titles whose description contains the word "love"
query = """
SELECT title, description
FROM netflix_titles
WHERE description LIKE '%love%';
"""
pd.read_sql(query, conn)

,title,description
0,Jeans,When the father of the man she loves insists t...
1,Love on the Spectrum,Finding love can be hard for anyone. For young...
2,Minsara Kanavu,A tangled love triangle ensues when a man fall...
3,Grown Ups,Mourning the loss of their beloved junior high...
4,Ankahi Kahaniya,"As big city life buzzes around them, lonely so..."
...,...,...
699,Winter of Our Dreams,"After the death of a long-ago lover, married p..."
700,Wrong No.,Two identical strangers pursue their respectiv...
701,Yaadein,Two young lovers set out to overcome the obsta...
702,Yaara O Dildaara,The patriarch of a wealthy family with one ind...


In [30]:
# Find the most recent movie released per country
query = """
SELECT country, MAX(release_year) AS latest_year
FROM netflix_titles
WHERE type = 'Movie'
GROUP BY country
ORDER BY latest_year DESC;
"""
pd.read_sql(query, conn)

,country,latest_year
0,Vietnam,2021
1,"United States, United Kingdom",2021
2,"United States, Japan",2021
3,"United States, India",2021
4,"United States, Canada",2021
...,...,...
647,West Germany,1977
648,"Poland,",1975
649,"United States, East Germany, West Germany",1971
650,"United States, Italy, United Kingdom, Liechten...",1965


In [31]:
# Count how many titles were released each year
query = """
SELECT release_year, COUNT(*) AS total
FROM netflix_titles
GROUP BY release_year
ORDER BY release_year DESC;
"""
pd.read_sql(query, conn)

,release_year,total
0,2021,592
1,2020,953
2,2019,1030
3,2018,1147
4,2017,1032
...,...,...
69,1945,4
70,1944,3
71,1943,3
72,1942,2
